In [94]:
import re

# import spacy
import pandas as pd

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords



from sklearn.feature_extraction.text import CountVectorizer


from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.dimensionality import BaseDimensionalityReduction

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\R1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
stopWords = stopwords.words('russian')

In [93]:
news = pd.read_csv("../data/raw/nnews.csv")

In [95]:
news.dropna(subset=['text'], inplace=True)

def clean_date(date_str):
    try:
        # Используем регулярное выражение для извлечения даты в формате YYYY-MM-DD
        match = re.search(r'(\d{4}-\d{2}-\d{2})', str(date_str))
        if match:
            return match.group(1)
        return date_str
    except:
        return date_str

news['date'] = news['date'].apply(clean_date)
news.shape

(1832, 4)

In [6]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
dim_model = UMAP(n_neighbors=15, n_components=10, min_dist=0.0, metric='cosine')
cluster_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

In [7]:
vectorizer_model = CountVectorizer(stop_words=stopWords)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = KeyBERTInspired()

In [8]:
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=dim_model,                     # Step 2 - Reduce dimensionality
  hdbscan_model=cluster_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [9]:
topics, probs = topic_model.fit_transform(news['text'])

In [10]:
news.isnull().sum()

text    0
date    0
dtype: int64

In [11]:
# Получаем эмбеддинги документов
embeddings = embedding_model.encode(news['text'].tolist(), show_progress_bar=True)
embeddings.shape  # Выводим размерность полученных эмбеддингов

dim_2d_model = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine')
zipped_embs = dim_2d_model.fit_transform(embeddings)
cluster_model.fit(zipped_embs)


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

HDBSCAN(min_cluster_size=15, prediction_data=True)

In [12]:
zipped_embs.shape

(1814, 2)

In [13]:
# Создаем более осмысленные названия для топиков
# Используем KeyBERT для генерации ключевых фраз из документов каждого топика

from keybert import KeyBERT

# Инициализируем модель KeyBERT
keybert_model = KeyBERT()

# Получаем информацию о топиках
topic_info = topic_model.get_topic_info()
topic_docs = {}

# Для каждого топика (кроме -1, который означает выбросы) получаем репрезентативные документы
for topic_id in topic_info[topic_info['Topic'] != -1]['Topic']:
    # Получаем документы для данного топика
    documents = topic_model.get_representative_docs(topic_id)
    topic_docs[topic_id] = ' '.join(documents)

# Создаем словарь для хранения новых названий топиков
topic_names = {}

# Для каждого топика генерируем ключевые фразы
for topic_id, doc in topic_docs.items():
    # Извлекаем ключевые фразы (2 слова) из документов топика
    keywords = keybert_model.extract_keywords(doc, keyphrase_ngram_range=(2, 2), stop_words=stopWords, top_n=1)
    
    if keywords:
        # Берем первую ключевую фразу как название топика
        topic_names[topic_id] = keywords[0][0]
    else:
        # Если не удалось извлечь фразу, используем оригинальное название
        words = topic_model.get_topic(topic_id)
        topic_names[topic_id] = f"Топик_{topic_id}_{words[0][0]}_{words[1][0]}"

# Переименовываем топики в модели
topic_model.set_topic_labels(topic_names)

# Выводим обновленную информацию о топиках
# print("Топики с новыми названиями:")
# display(topic_model.get_topic_info()[1:11])


In [14]:
topic_model.get_topic_info().to_csv('../data/interim/topic_info.csv', index=False)

In [1]:
import pandas as pd

data = pd.read_csv('../data/interim/topic_info.csv')

In [22]:
import json

# Проверяем содержимое перед парсингом
print(data['Representative_Docs'].iloc[0])

# Исправляем ошибку парсинга JSON - возможно, строка требует предварительной обработки
try:
    # Пробуем очистить строку от лишних символов и заменить одинарные кавычки на двойные
    cleaned_json = data['Representative_Docs'].iloc[0].replace("'", '"')
    parsed_data = json.loads(cleaned_json)
    print("Успешно распарсили JSON")
    print(parsed_data)
except json.JSONDecodeError as e:
    print(f"Ошибка парсинга JSON: {e}")
    # Альтернативный подход - использовать ast.literal_eval для парсинга Python литералов
    import ast
    try:
        parsed_data = ast.literal_eval(data['Representative_Docs'].iloc[0])
        print("Успешно распарсили с помощью ast.literal_eval")
        print(parsed_data)
    except:
        print("Не удалось распарсить данные")

['Глава МВД Украины Арсен Аваков заявил, что Киев и Москва могут прийти к компромиссу в вопросе передачи Украине контроля над границей в Донбассе. Интервью с министром опубликовано на странице издания «Громадське» в Twitter. По его словам, в первое время контроль над границей могут осуществлять не погранвойска, а украинская полиция вместе с «представителями территориальных общин». При этом он указал, что это станет возможным только после того, как вооруженные формирования покинут территории самопровозглашенных республик. Аваков считает, что такой «переходный период» может продолжаться вплоть до года, но Украина «готова это пройти». При этом он указал, что участники «нормандского саммита» не дали согласия на такой вариант, а президент России Владимир Путин «не готов вернуть границу». Говоря о возможном компромиссе, Аваков отметил, что Киев может получить контроль над границей «не за месяц до местных выборов, а за два дня». Ранее президент Украины Владимир Зеленский заявил о необходимост

In [23]:
parsed_data

['Глава МВД Украины Арсен Аваков заявил, что Киев и Москва могут прийти к компромиссу в вопросе передачи Украине контроля над границей в Донбассе. Интервью с министром опубликовано на странице издания «Громадське» в Twitter. По его словам, в первое время контроль над границей могут осуществлять не погранвойска, а украинская полиция вместе с «представителями территориальных общин». При этом он указал, что это станет возможным только после того, как вооруженные формирования покинут территории самопровозглашенных республик. Аваков считает, что такой «переходный период» может продолжаться вплоть до года, но Украина «готова это пройти». При этом он указал, что участники «нормандского саммита» не дали согласия на такой вариант, а президент России Владимир Путин «не готов вернуть границу». Говоря о возможном компромиссе, Аваков отметил, что Киев может получить контроль над границей «не за месяц до местных выборов, а за два дня». Ранее президент Украины Владимир Зеленский заявил о необходимост

In [ ]:
topic_model.get_topic_info()["CustomName"]

In [56]:
# Визуализация кластеров новостей в 2D пространстве с помощью plotly
import plotly.express as px
import pandas as pd
import numpy as np

# Получаем координаты документов в 2D пространстве
embeddings_2d = zipped_embs

# Получаем данные о документах
doc_info = topic_model.get_document_info(news['text'].tolist())

# Создаем DataFrame для визуализации
plot_df = pd.DataFrame({
    'x': embeddings_2d.embedding_x,
    'y': embeddings_2d.embedding_y,
    'topic': embeddings_2d.topic,
    'text': doc_info.Document,
    'topic_name': doc_info.Name
})

# Создаем цветовую схему
colors = px.colors.qualitative.Plotly

# Создаем интерактивную визуализацию
fig = px.scatter(
    plot_df, 
    x='x', 
    y='y', 
    color='topic_name',
    hover_data=['text'],
    title='Кластеризация новостей по темам',
    color_discrete_sequence=colors,
    opacity=0.7,
    size_max=10
)

# Настраиваем внешний вид графика
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    legend_title_text='Темы',
    xaxis_title="",
    yaxis_title="",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False),
    plot_bgcolor='white'
)

# Отображаем график
fig.show()



NameError: name 'df' is not defined

In [44]:
topic_model.visualize_barchart(top_n_topics=30, n_words=10, title='Топ слов по темам')